# Pipeline — chạy full `qshield-pipeline all` (5 chặng, không mock)

Gọi `qshield_pipeline.run.run_all()` — **an toàn gọi trực tiếp trong notebook**, khác
`quantum_solve.ipynb`: bản thân `run_all()` đã tự cô lập MỖI chặng bằng subprocess riêng
(`_run_stage_subprocess`, xem `packages/pipeline/src/qshield_pipeline/run.py`), nên tiến trình
notebook không bao giờ import trực tiếp `qshield_quantum` (qiskit) — không có rủi ro segfault hay
`os._exit` giết kernel như ở notebook trước.

**Đây là lần đầu tiên `qshield-pipeline all` chạy thật, đủ 5 chặng, không mock.** Test của
`packages/pipeline` từ trước tới giờ đều monkeypatch `_run_stage_subprocess` (nhanh nhưng không
xác nhận pipeline THẬT chạy được) — notebook này là xác nhận thật đầu tiên.

**5 chặng:** `data` (fetch lại thật qua mạng) → `regime` → `scenarios` (500 kịch bản, mặc định
`demo_fast`) → `risk` (`effects`, PROVISIONAL transaction cost) → `optimize`
(`qshield-quantum solve`, chấm true CVaR).

⚠️ Chặng `data` fetch lại thật — chậm hơn 4 chặng còn lại cộng lại. Không có timeout nội bộ trong
`run_all()`; nếu mạng treo, dừng kernel thủ công (Kernel → Interrupt).

In [1]:
import os
import time
from pathlib import Path

import yaml
from qshield_contracts.config import Config


def _find_project_root(marker: str = "CLAUDE.md") -> Path:
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"Không tìm thấy {marker} từ {p} trở lên — notebook phải nằm trong repo QSHIELD."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"
cfg = Config.load(CONFIG_PATH)
cfg["artifacts"]["mode"]

'dev'

## Bước 0: Config PROVISIONAL cho transaction cost (chặng `risk`/`optimize` cần)

Giống hệt `risk_effects.ipynb`/`quantum_solve.ipynb` — KHÔNG sửa `configs/risk.yaml` gốc, chỉ ghi
bản resolved riêng cho lần chạy này.

In [2]:
from qshield_contracts.paths import ArtifactPaths

PROVISIONAL_TRANSACTION_COST = {
    "fee": 0.0015,
    "spread": 0.0010,
    "liquidity_penalty": 0.0005,
}
PROVISIONAL_WEIGHT_SUM_TOLERANCE = 1e-6

resolved_cfg = dict(cfg)
resolved_cfg["transaction_cost"] = PROVISIONAL_TRANSACTION_COST
resolved_cfg["weight_sum_tolerance"] = PROVISIONAL_WEIGHT_SUM_TOLERANCE

# Giống 3 notebook trước — ghi resolved config vào run_root (qua ArtifactPaths, CLAUDE.md quy tắc
# 8), không lạc trong configs/.
paths = ArtifactPaths(resolved_cfg, run_id=None)
paths.run_root.mkdir(parents=True, exist_ok=True)
resolved_config_path = paths.run_root / "_pipeline_notebook_resolved_config.yaml"
resolved_config_path.write_text(
    yaml.safe_dump(resolved_cfg, allow_unicode=True), encoding="utf-8"
)
print(f"Config đã resolve → {resolved_config_path}")
print(
    "⚠️  NON_BASELINE_RUN — transaction_cost là placeholder, chưa được Phúc/Ngọc duyệt."
)

Config đã resolve → artifacts/dev/_pipeline_notebook_resolved_config.yaml
⚠️  NON_BASELINE_RUN — transaction_cost là placeholder, chưa được Phúc/Ngọc duyệt.


## Bước 1: Chạy `run_all()` — 5 chặng tuần tự, fail-fast

In [3]:
from qshield_pipeline.run import StageError, run_all

RUN_START = time.perf_counter()
try:
    run_id = run_all(resolved_config_path, mock=False)
except StageError:
    total_seconds = time.perf_counter() - RUN_START
    print(f"[timing] Dừng sau {total_seconds:.1f}s")
    raise
total_seconds = time.perf_counter() - RUN_START
print(f"[pipeline] OK — toàn bộ 5 chặng PASS (run_id={run_id or 'dev'})")
print(
    f"[timing] Full pipeline (data→regime→scenarios→risk→optimize): {total_seconds:.1f}s"
)

Bắt đầu pipeline — run_id=None, mock=False, 5 chặng: data, regime, scenarios, risk, optimize


[1/5] Data (qshield-data build)


INFO:qshield_data.sources.fetch:Downloading 8 tickers from 2016-01-01 to 2026-07-31 (route: yahoo/dnse theo universe['data_source'])


status
OK    8


INFO:qshield_data.sources.fetch:Trying VN-Index from vnstock (source=VCI)...


...

/Users/mac/Documents/QSHIELD/.venv/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/mac/Documents/QSHIELD/.venv/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


/Users/mac/Documents/QSHIELD/.venv/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/mac/Documents/QSHIELD/.venv/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/Users/mac/Documents/QSHIELD/.venv/lib/python3.14/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


/Users/mac/Documents/QSHIELD/.venv/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/Users/mac/Documents/QSHIELD/.venv/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


QAOA: chạy xong 10 seed.
QAOA KHÔNG thắng classical baseline (greedy theo g) — báo cáo trung thực theo CLAUDE.md quy tắc 18, không diễn giải có lợi cho QAOA.
True CVaR (qshield_risk.evaluate): before=0.110032 after=0.100704 (improves_cvar)
Pipeline hoàn tất — toàn bộ 5 chặng PASS.


[quantum] OK — bitstring=01011000 → artifacts/dev/optimization/qaoa_result.json
[pipeline] OK — toàn bộ 5 chặng PASS (run_id=dev)
[timing] Full pipeline (data→regime→scenarios→risk→optimize): 130.6s


## Bước 2: Checklist artifact cả 5 chặng

In [4]:
artifact_checks = [
    PROJECT_ROOT / "data" / "processed" / "returns.parquet",
    PROJECT_ROOT / "artifacts" / "dev" / "regime" / "regime_daily.parquet",
    PROJECT_ROOT / "artifacts" / "dev" / "scenarios" / "stress_scenarios.npz",
    PROJECT_ROOT / "artifacts" / "dev" / "risk" / "action_effects.csv",
    PROJECT_ROOT / "artifacts" / "dev" / "optimization" / "qaoa_result.json",
    PROJECT_ROOT / "artifacts" / "dev" / "optimization" / "benchmark.json",
]
for path in artifact_checks:
    status = "✓" if path.exists() else "✗ THIẾU"
    print(f"{status} {path.relative_to(PROJECT_ROOT)}")

✓ data/processed/returns.parquet
✓ artifacts/dev/regime/regime_daily.parquet
✓ artifacts/dev/scenarios/stress_scenarios.npz
✓ artifacts/dev/risk/action_effects.csv
✓ artifacts/dev/optimization/qaoa_result.json
✓ artifacts/dev/optimization/benchmark.json


## Xong

Toàn bộ `NON_BASELINE_RUN` (transaction cost/penalty còn PROVISIONAL, xem 2 notebook trước).
`artifacts/dev/logs.txt` có log của cả 5 chặng nối tiếp (cùng ghi vào một `run_root` ở `dev`
mode) — đọc để thấy đủ cảnh báo PROVISIONAL từng chặng đã in ra lúc chạy.

`artifacts/dev/_pipeline_notebook_resolved_config.yaml` chỉ để chạy thử, không phải config chính
thức của repo — không cần commit.